# Gaze Tracking — Yaw, Pitch & Crosshair
This notebook explains and demonstrates the hybrid gaze tracking system used in ARIA 2.0.

**How it works:**
- **Yaw (left/right)** — blend of head turn + iris X offset
- **Pitch (up/down)** — head geometry ratio (nose_to_chin / forehead_to_nose)
- **Crosshair** — mapped to screen using calibration or raw estimate

Run each cell in order.

## Cell 1 — Imports
We only need OpenCV, MediaPipe, and NumPy. No external gaze model required.

In [56]:
import cv2
import numpy as np
import mediapipe as mp
import time

## Cell 2 — MediaPipe Face Landmarker Setup
We use MediaPipe's Tasks API in VIDEO mode.

- `num_faces = 1` — we only need one face
- `VIDEO` mode — requires a monotonically increasing timestamp per frame
- Returns 478 landmarks including **iris landmarks (468–477)**

In [57]:
import os

BASE_DIR = os.getcwd()  # assumes face_landmarker.task is in the same folder

BaseOptions           = mp.tasks.BaseOptions
FaceLandmarker        = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode     = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options = BaseOptions(model_asset_path=os.path.join(BASE_DIR, '../Mini-Project-2/face_landmarker.task')),
    running_mode = VisionRunningMode.VIDEO,
    num_faces    = 1
)

landmarker = FaceLandmarker.create_from_options(options)
print('Landmarker ready!')

Landmarker ready!


## Cell 3 — Landmark Indices
We use specific landmark indices for head pose and iris tracking.

```
HEAD POSE:
  4   = Nose tip       — moves left/right with head turn
  10  = Forehead       — top of face
  152 = Chin           — bottom of face
  234 = Left cheek     — leftmost point of face
  454 = Right cheek    — rightmost point of face

IRIS (left eye):
  468 = Left iris center
  33  = Left eye left corner
  133 = Left eye right corner

IRIS (right eye):
  473 = Right iris center
  362 = Right eye left corner
  263 = Right eye right corner
```

In [58]:
# Head pose landmarks
NOSE_TIP = 4
FOREHEAD = 10
CHIN     = 152
L_CHEEK  = 234
R_CHEEK  = 454

# Iris landmarks
L_IRIS      = 468
L_EYE_LEFT  = 33
L_EYE_RIGHT = 133
R_IRIS      = 473
R_EYE_LEFT  = 362
R_EYE_RIGHT = 263

# Blend weights for yaw
HEAD_WEIGHT = 0
IRIS_WEIGHT = 1

# Smoothing
SMOOTH_WINDOW = 10
yaw_history   = []
pitch_history = []

print('Indices and constants set!')

Indices and constants set!


## Cell 4 — Head Pose Functions

### `get_head_yaw(landmarks)`
Measures how far the nose tip is from the horizontal center of the face.
- Face center X = average of left and right cheek X
- Nose offset normalized by face width
- Result: negative = looking left, positive = looking right

### `get_head_pitch(landmarks)`
Uses the ratio of **nose-to-chin distance** vs **forehead-to-nose distance**.
- When looking **up**: chin comes forward, ratio increases (> 1.3)
- When looking **down**: forehead dominates, ratio decreases (< 1.0)
- This is more reliable than raw Y position because it's self-normalizing

In [59]:
def get_head_yaw(landmarks):
    """
    Returns horizontal head turn as a normalized float.
    Negative = left, Positive = right.
    Typical range: -0.4 to 0.4
    """
    nose    = landmarks[NOSE_TIP]
    l_cheek = landmarks[L_CHEEK]
    r_cheek = landmarks[R_CHEEK]

    face_center_x = (l_cheek.x + r_cheek.x) / 2
    face_width    =  r_cheek.x - l_cheek.x

    if face_width < 1e-6:
        return None

    return (nose.x - face_center_x) / face_width


def get_head_pitch(landmarks):
    """
    Returns vertical head tilt as a ratio.
    Higher value = looking up, Lower = looking down.
    Typical range: 0.9 (down) to 1.5 (up)
    """
    nose = landmarks[NOSE_TIP]
    fore = landmarks[FOREHEAD]
    chin = landmarks[CHIN]

    nose_to_chin     = abs(nose.y - chin.y)
    forehead_to_nose = abs(fore.y - nose.y)

    if forehead_to_nose < 1e-6:
        return None

    return nose_to_chin / forehead_to_nose


def get_iris_x_offset(landmarks):
    """
    Returns average iris X offset across both eyes.
    0.0 = center, negative = left, positive = right.
    Typical range: -0.15 to 0.15
    """
    def eye_offset(iris_idx, left_idx, right_idx):
        iris  = landmarks[iris_idx]
        left  = landmarks[left_idx]
        right = landmarks[right_idx]
        eye_w = abs(right.x - left.x)
        if eye_w < 1e-6:
            return None
        return ((iris.x - left.x) / eye_w) - 0.5

    l = eye_offset(L_IRIS, L_EYE_LEFT, L_EYE_RIGHT)
    r = eye_offset(R_IRIS, R_EYE_LEFT, R_EYE_RIGHT)

    if l is None and r is None: return None
    if l is None: return r
    if r is None: return l
    return (l + r) / 2.0


def smooth(value, history):
    """
    Adds value to history buffer and returns smoothed average.
    Reduces jitter from frame-to-frame noise.
    """
    history.append(value)
    if len(history) > SMOOTH_WINDOW:
        history.pop(0)
    return sum(history) / len(history)


print('Head pose functions ready!')

Head pose functions ready!


## Cell 5 — Live Capture
Runs the webcam and displays:
- **Yaw / Pitch values** — bottom left
- **Gaze crosshair** — yellow dot mapped to screen position
- **Iris position** — small dot on each iris
- **Landmark dots** — nose, forehead, chin, cheeks for reference

Press **Q** to quit.

In [60]:
WHITE  = (255, 255, 255)
GREEN  = (0, 255, 0)
YELLOW = (0, 255, 255)
RED    = (0, 0, 255)
CYAN   = (255, 255, 0)

cap = cv2.VideoCapture(0)
yaw_history   = []
pitch_history = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w  = frame.shape[:2]

    # ── Run MediaPipe ────────────────────────────────────────────────
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    result    = landmarker.detect_for_video(mp_image, int(time.time() * 1000))

    if result.face_landmarks:
        lm = result.face_landmarks[0]
        
        nose_tip = lm[NOSE_TIP]
        forehead = lm[FOREHEAD]
        chin = lm[CHIN]
        l_cheek = lm[L_CHEEK]
        r_cheek = lm[R_CHEEK]

        # ── Compute yaw, pitch, iris ─────────────────────────────────
        head_yaw  = get_head_yaw(lm)
        head_pitch = get_head_pitch(lm)
        iris_x    = get_iris_x_offset(lm)

        # Blend yaw: head + iris
        if head_yaw is not None and iris_x is not None:
            raw_yaw = head_yaw * HEAD_WEIGHT + iris_x * IRIS_WEIGHT
        else:
            raw_yaw = head_yaw

        # Smooth
        yaw   = smooth(raw_yaw,    yaw_history)   if raw_yaw    is not None else None
        pitch = smooth(head_pitch, pitch_history)  if head_pitch is not None else None

        # ── Draw landmark reference dots ─────────────────────────────
        for idx, color in [
            (NOSE_TIP, GREEN),
            (FOREHEAD, CYAN),
            (CHIN,     CYAN),
            (L_CHEEK,  RED),
            (R_CHEEK,  RED),
        ]:
            px = int(lm[idx].x * w)
            py = int(lm[idx].y * h)
            cv2.circle(frame, (px, py), 5, color, -1)
            
        # face lines
        cv2.line(frame, (int(forehead.x*w), int(forehead.y*h)), (int(nose_tip.x*w), int(nose_tip.y*h)), GREEN, 1)
        cv2.line(frame, (int(nose_tip.x*w), int(nose_tip.y*h)), (int(chin.x*w), int(chin.y*h)), GREEN, 1)
        cv2.line(frame, (int(l_cheek.x*w), int(l_cheek.y*h)), (int(nose_tip.x*w), int(nose_tip.y*h)), GREEN, 1)
        cv2.line(frame, (int(nose_tip.x*w), int(nose_tip.y*h)), (int(r_cheek.x*w), int(r_cheek.y*h)), GREEN, 1)
        
        

        # ── Draw iris dots ───────────────────────────────────────────
        for iris_idx in [L_IRIS, R_IRIS]:
            px = int(lm[iris_idx].x * w)
            py = int(lm[iris_idx].y * h)
            cv2.circle(frame, (px, py), 4, YELLOW, -1)

        # ── Draw gaze crosshair ──────────────────────────────────────
        if yaw is not None and pitch is not None:
            # Map to screen — raw estimate (no calibration in this notebook)
            cx = int(w * 0.5 + yaw   * w * 2.0)
            cy = int(h * 0.5 - (pitch - 1.2) * h * 2.0)
            cx = max(20, min(w - 20, cx))
            cy = max(20, min(h - 20, cy))

            cv2.circle(frame, (cx, cy), 18, YELLOW, 2)
            cv2.circle(frame, (cx, cy), 4,  YELLOW, -1)
            cv2.line(frame, (cx - 25, cy), (cx + 25, cy), YELLOW, 1)
            cv2.line(frame, (cx, cy - 25), (cx, cy + 25), YELLOW, 1)
            

        # ── Display values ───────────────────────────────────────────
        lines = [
            (f'Head Yaw:   {head_yaw:.3f}',  WHITE),
            (f'Iris X:     {iris_x:.3f}',    YELLOW),
            (f'Blended Yaw:{yaw:.3f}',        GREEN),
            (f'Pitch:      {pitch:.3f}',      CYAN),
        ] if (head_yaw and iris_x and yaw and pitch) else []

        for i, (text, color) in enumerate(lines):
            cv2.putText(frame, text, (20, h - 20 - i * 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1)

    else:
        cv2.putText(frame, 'No face detected', (20, h - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, RED, 1)

    cv2.imshow('Gaze Tracker — Yaw & Pitch', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print('Done!')

Done!
